# NLTK

In [63]:
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('maxent_ne_chunker_tab')
# nltk.download('words')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('gutenberg')
# nltk.download('reuters')
# nltk.download('omw-1.4')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score


import re
import nltk
from nltk.text import Text
from nltk.corpus import words, wordnet as wn, stopwords, gutenberg, reuters
from nltk import word_tokenize, sent_tokenize, ne_chunk, pos_tag, trigrams
from nltk.tokenize import WordPunctTokenizer, TreebankWordTokenizer, RegexpTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams, bigrams
from nltk.probability import FreqDist
from nltk.wsd import lesk

from collections import defaultdict

In [7]:
text= "The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!"

words=text.split() #Also tokenizing the text from spaces or a specific character

## Tokenization

In [73]:
# Tokenization
# downloaded punkt_tab for this
print("Original Sentence: ", text)
word_tokenized=word_tokenize(text)
print("After word tokenizing: ",word_tokenized)
sent_tokenized=sent_tokenize(text)
print("After sentence tokenizing: ",sent_tokenized)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', 'dog', "'s", '.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']
After sentence tokenizing:  ["The quick brown foxes are jumping over the l@zy dog's.", 'The sun is shining which is a fantastic sight!']


In [74]:
tokenizer= WordPunctTokenizer()
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', 'dog', "'", 's', '.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']


In [75]:
tokenizer=TreebankWordTokenizer()
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', '@', 'zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight', '!']


In [77]:
tokenizer= RegexpTokenizer(r'\w+')
print("Original Sentence: ", text)
tokenized_text=tokenizer.tokenize(text)
print("After word punct tokenizing: ",tokenized_text)

Original Sentence:  The quick brown foxes are jumping over the l@zy dog's. The sun is shining which is a fantastic sight!
After word punct tokenizing:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', 'zy', 'dog', 's', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


## Stemming

In [54]:
#Stemming 
print("Original words: ", words)
porter= PorterStemmer() # stems word by word, so using loop
stemmed_words= [porter.stem(word) for word in words] #also converts to lowercase
print("After stemming: ",stemmed_words)

print(f"Example: \n 'Playing' -> {porter.stem('playing')} \n 'played' -> {porter.stem('played')} \n 'plays' -> {porter.stem('plays')} \n 'communication' -> {porter.stem('communication')} ")

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After stemming:  ['the', 'quick', 'brown', 'fox', 'are', 'jump', 'over', 'the', 'lazi', 'dogs.', 'the', 'sun', 'is', 'shine', 'which', 'is', 'a', 'fantast', 'sight.']
Example: 
 'Playing' -> play 
 'played' -> play 
 'plays' -> play 
 'communication' -> commun 


## Lemmetization

In [55]:
# Lemmetization

print("Original words: ", words)
wnl= WordNetLemmatizer() #slower than stemming, also use loop because it accepts single word
lemmatized_words_without_pos= [wnl.lemmatize(word) for word in words]
print("After lemmatizing without POS: ",lemmatized_words_without_pos)

lemmatized_words_with_pos= [wnl.lemmatize(word, pos='v') for word in words] #used verb for every word for simplicity, need to pass POS accordingly for each word
print("After lemmatizing with POS: ",lemmatized_words_with_pos)

print(f"Example: \n 'Playing' -> {wnl.lemmatize('playing', pos='v')} \n 'played' -> {wnl.lemmatize('played', pos='v')} \n 'plays' -> {wnl.lemmatize('plays', pos='v')} \n 'communication' -> {wnl.lemmatize('communication')} ")

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After lemmatizing without POS:  ['The', 'quick', 'brown', 'fox', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After lemmatizing with POS:  ['The', 'quick', 'brown', 'fox', 'be', 'jump', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'be', 'shin', 'which', 'be', 'a', 'fantastic', 'sight.']
Example: 
 'Playing' -> play 
 'played' -> play 
 'plays' -> play 
 'communication' -> communication 


## Stop Words

In [ ]:
# Stop Words

stop_words = set(stopwords.words('english'))
lower_words= [word.lower() for word in words] # used this because all stopwords are in lowercase. It will not accept 'The'
print("Original words: ", lower_words)
no_stop_words= [word for word in lower_words if word not in stop_words]
print("After removing stop words: ",no_stop_words)

Original words:  ['the', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'the', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing stop words:  ['quick', 'brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'sun', 'shining', 'fantastic', 'sight!']


In [127]:
# Custom stopwords

print(f"Default Stopwords: {len(stop_words)} words")

custom_stopwords = {'sun', 'quick'}
custom_stopwords = stop_words.union(custom_stopwords)

print(f"Total Stopwords including custom: {len(custom_stopwords)} words\n")

print("After removing standard stop words ", no_stop_words)
no_custom_stop_words= [word for word in lower_words if word not in custom_stopwords]
print("After removing custom stop words: ",no_custom_stop_words)

Default Stopwords: 198 words
Total Stopwords including custom: 200 words

After removing standard stop words  ['quick', 'brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'sun', 'shining', 'fantastic', 'sight!']
After removing custom stop words:  ['brown', 'foxes', 'jumping', 'l@zy', "dog's.", 'shining', 'fantastic', 'sight!']


## Punctuation removal

In [110]:
# Punctuation Removal

print("Original words: ", words)
no_punct_words= [re.sub(r'[^\w\s]','',words) for words in words if re.sub(r'[^\w\s]','',words)]
print("After removing punctuations: ",no_punct_words)


Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing punctuations:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lzy', 'dogs', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


In [113]:
tokenizer = RegexpTokenizer(r'\w+')

print("Original words: ", words)
no_punct_words= tokenizer.tokenize(text)
print("After removing punctuations: ",no_punct_words)

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l@zy', "dog's.", 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight!']
After removing punctuations:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'l', 'zy', 'dog', 's', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight']


## POS Tagging

In [56]:
# Part Of Speech (POS) tagging
# Downloaded averaged_perceptron_tagger_eng for this
print("Original words: ", words)
pos_tagged_words= pos_tag(words)
print("After POS tagging: ",pos_tagged_words)


Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.']
After POS tagging:  [('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('foxes', 'NNS'), ('are', 'VBP'), ('jumping', 'VBG'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dogs.', 'NN'), ('The', 'DT'), ('sun', 'NN'), ('is', 'VBZ'), ('shining', 'VBG'), ('which', 'WDT'), ('is', 'VBZ'), ('a', 'DT'), ('fantastic', 'JJ'), ('sight.', 'NN')]


## NER

In [59]:
# Named Entity Recognition (NER)
#downloaded 'maxent_ne_chunker_tab' and 'words' for this
words1= words+['Narendra']+['Rahul']
pos_tagged_words= pos_tag(words1)
print("Original words: ", words1)
ne_chunked_words= ne_chunk(pos_tagged_words)
print("After NER: ",ne_chunked_words)

Original words:  ['The', 'quick', 'brown', 'foxes', 'are', 'jumping', 'over', 'the', 'lazy', 'dogs.', 'The', 'sun', 'is', 'shining', 'which', 'is', 'a', 'fantastic', 'sight.', 'Narendra', 'Rahul']
After NER:  (S
  The/DT
  quick/JJ
  brown/NN
  foxes/NNS
  are/VBP
  jumping/VBG
  over/IN
  the/DT
  lazy/JJ
  dogs./NN
  The/DT
  sun/NN
  is/VBZ
  shining/VBG
  which/WDT
  is/VBZ
  a/DT
  fantastic/JJ
  sight./NN
  (PERSON Narendra/NNP Rahul/NNP))


## Concordance

In [85]:
conc_text=Text(word_tokenize(text))
conc_text.concordance("jumping")

Displaying 1 of 1 matches:
          The quick brown foxes are jumping over the l @ zy dog 's . The sun is


## Correcting Words

In [98]:
# doownloaded words for this.
incorrect_words=['happpye', 'azmaingi', 'intelliengxt']
correct_words=words.words()

for word in incorrect_words:
    temp = [(jaccard_distance(set(ngrams(word, 2)), set(ngrams(w, 2))),w) for w in correct_words if w[0]==word[0]]
    print(sorted(temp, key = lambda val:val[0])[0][1])

happy
amazing
intelligent


## Corpuses

In [135]:
print(gutenberg.fileids()) #prints all the available literatures available in the gutenberg corpus
bible=gutenberg.words('bible-kjv.txt')
print(bible[:20])

['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']
['[', 'The', 'King', 'James', 'Bible', ']', 'The', 'Old', 'Testament', 'of', 'the', 'King', 'James', 'Bible', 'The', 'First', 'Book', 'of', 'Moses', ':']


In [153]:
synonyms= wordnet.synsets('good') # provides similar words and sometimes the same word with multiple meanings
print(synonyms)
print(synonyms[0].lemmas())
print(synonyms[1].lemmas()[0].antonyms())
print(synonyms[0].definition())
print(synonyms[0].examples())

[Synset('good.n.01'), Synset('good.n.02'), Synset('good.n.03'), Synset('commodity.n.01'), Synset('good.a.01'), Synset('full.s.04'), Synset('good.a.03'), Synset('estimable.s.01'), Synset('beneficial.s.01'), Synset('good.s.04'), Synset('good.s.05'), Synset('adept.s.01'), Synset('good.s.07'), Synset('dear.s.02'), Synset('dependable.s.03'), Synset('good.s.10'), Synset('good.s.11'), Synset('effective.s.03'), Synset('good.s.13'), Synset('good.s.14'), Synset('good.s.15'), Synset('good.s.16'), Synset('good.s.17'), Synset('good.s.18'), Synset('good.s.19'), Synset('well.r.01'), Synset('thoroughly.r.02')]
[Lemma('good.n.01.good')]
[Lemma('evil.n.03.evil')]
benefit
['for your own good', "what's the good of worrying?"]


In [157]:
sample = gutenberg.words('bible-kjv.txt')
fdist = FreqDist(sample)
print(fdist.most_common(10))

[(',', 70509), ('the', 62103), (':', 43766), ('and', 38847), ('of', 34480), ('.', 26160), ('to', 13396), ('And', 12846), ('that', 12576), ('in', 12331)]


## N-gram language modelling

In [9]:
# Creating Bigrams

bigram= bigrams(words)
for i,j in bigram:
    print(i,j)

The quick
quick brown
brown foxes
foxes are
are jumping
jumping over
over the
the l@zy
l@zy dog's.
dog's. The
The sun
sun is
is shining
shining which
which is
is a
a fantastic
fantastic sight!


In [ ]:
#predicting next word

words = nltk.word_tokenize(' '.join(reuters.words()))
tri_grams = trigrams(words)

model = defaultdict(lambda: defaultdict(lambda: 0))
for w1, w2, w3 in tri_grams:
    model[(w1, w2)][w3] += 1
    
for w1_w2 in model:
    total_count = float(sum(model[w1_w2].values()))
    for w3 in model[w1_w2]:
        model[w1_w2][w3] /= total_count
        
def predict_next_word(w1, w2):
    next_word_probs = model[w1, w2]
    if next_word_probs:
        return max(next_word_probs, key=next_word_probs.get)
    else:
        return "No prediction available"

In [201]:
word1= input("Enter the first word: ")
word2= input("Enter the second word: ")
print(f"Predicting next word:\n {word1} {word2}->{predict_next_word(word1, word2)}")

Predicting next word:
 prices increase->commodity


## Basic Sentiment Analysis

In [50]:
df= pd.read_csv(r'D:\AppStoneLab\NLP\Datasets\training.1600000.processed.noemoticon.csv.zip', encoding='latin-1', names=['target', 'ids', 'date', 'flag', 'user', 'text'])
df.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [20]:
tweet= df['text'][0]
tokens= word_tokenize(tweet)
tagged= pos_tag(tokens)
print(tweet)
print(tokens)
print(tagged)

@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D
['@', 'switchfoot', 'http', ':', '//twitpic.com/2y1zl', '-', 'Awww', ',', 'that', "'s", 'a', 'bummer', '.', 'You', 'shoulda', 'got', 'David', 'Carr', 'of', 'Third', 'Day', 'to', 'do', 'it', '.', ';', 'D']
[('@', 'JJ'), ('switchfoot', 'NN'), ('http', 'NN'), (':', ':'), ('//twitpic.com/2y1zl', 'JJ'), ('-', ':'), ('Awww', 'NN'), (',', ','), ('that', 'WDT'), ("'s", 'VBZ'), ('a', 'DT'), ('bummer', 'NN'), ('.', '.'), ('You', 'PRP'), ('shoulda', 'VBP'), ('got', 'VBD'), ('David', 'NNP'), ('Carr', 'NNP'), ('of', 'IN'), ('Third', 'NNP'), ('Day', 'NNP'), ('to', 'TO'), ('do', 'VB'), ('it', 'PRP'), ('.', '.'), (';', ':'), ('D', 'NNP')]


In [23]:
sentence = "He went to the bank to deposit money."
tokens = word_tokenize(sentence)
sense = lesk(tokens, 'bank')
print("Best sense:", sense)
print("Definition:", sense.definition())

Best sense: Synset('depository_financial_institution.n.01')
Definition: a financial institution that accepts deposits and channels the money into lending activities


In [31]:
dog = wn.synsets('good', pos=wn.ADJ)[0]
bad = wn.synsets('bad', pos=wn.ADJ)[0]

similarity = dog.wup_similarity(bad)
print(f"Semantic Similarity (Wu-Palmer): {similarity}")

Semantic Similarity (Wu-Palmer): 0.5


In [33]:
tree = ne_chunk(tagged)
print(tree)

(S
  @/JJ
  switchfoot/NN
  http/NN
  :/:
  //twitpic.com/2y1zl/JJ
  -/:
  (GPE Awww/NN)
  ,/,
  that/WDT
  's/VBZ
  a/DT
  bummer/NN
  ./.
  You/PRP
  shoulda/VBP
  got/VBD
  (PERSON David/NNP Carr/NNP)
  of/IN
  (PERSON Third/NNP)
  Day/NNP
  to/TO
  do/VB
  it/PRP
  ./.
  ;/:
  D/NNP)


In [46]:
def semantic_analyze(text):
    tokenized= word_tokenize(text)
    tagged= pos_tag(tokenized)
    entity_chunks= ne_chunk(tagged)
    # print("Original:", text)
    # print("Tokens:", tokenized)
    # print("Named Entity tree:", entity_chunks)
    
    for word in tokenized:
        synsets= wn.synsets(word)
        if synsets:
            print(f"\nWord: {word}")
            for syn in synsets:
                print(f"    ->{syn.name()}: {syn.definition()}")
    print("\n")

In [47]:
semantic_analyze(df['text'][2])


Word: I
    ->iodine.n.01: a nonmetallic element belonging to the halogens; used especially in medicine and photography and in dyes; occurs naturally only in combination in small quantities (as in sea water or rocks)
    ->one.n.01: the smallest whole number or a numeral representing this number
    ->i.n.03: the 9th letter of the Roman alphabet
    ->one.s.01: used of a single unit or thing; not two or more

Word: dived
    ->dive.v.01: drop steeply
    ->dive.v.02: plunge into water
    ->dive.v.03: swim under water

Word: many
    ->many.a.01: a quantifier that can be used with count nouns and is often preceded by `as' or `too' or `so' or `that'; amounting to a large but indefinite number

Word: times
    ->times.n.01: a more or less definite period of time now or previously present
    ->multiplication.n.03: an arithmetic operation that is the inverse of division; the product of two numbers is computed
    ->time.n.01: an instance or single occasion for some event
    ->time.n.02:

## Text Classification

In [69]:
#Text Classification

df= pd.read_csv(r'D:\AppStoneLab\NLP\Datasets\training.1600000.processed.noemoticon.csv.zip', encoding='latin-1', names=['target', 'ids', 'date', 'flag', 'user', 'text'])
df['target'].value_counts()

target
0    800000
4    800000
Name: count, dtype: int64

In [62]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(tokens)

df['cleaned text']= df['text'].apply(preprocess)
df.drop(['ids', 'date', 'flag', 'user', 'text'], axis=1, inplace=True)
df.head()

,target,cleaned text
0,0,switchfoot http awww bummer shoulda got david ...
1,0,upset ca update facebook texting might cry res...
2,0,kenichan dived many times ball managed save re...
3,0,whole body feels itchy like fire
4,0,nationwideclass behaving mad ca see


In [67]:
vectorizer=TfidfVectorizer()
X= vectorizer.fit_transform(df['cleaned text'])
y= df['target'].map({0:'negative',2:'neutral', 4:'positive'})


In [70]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state=100)

model= MultinomialNB()
model.fit(X_train, y_train)
y_pred= model.predict(X_test)
cr= classification_report(y_test, y_pred)
print("Classification Report:\n",cr)
accuracy= accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Classification Report:
               precision    recall  f1-score   support

    negative       0.75      0.79      0.77    200237
    positive       0.78      0.74      0.76    199763

    accuracy                           0.76    400000
   macro avg       0.76      0.76      0.76    400000
weighted avg       0.76      0.76      0.76    400000

Accuracy: 0.7644575


In [110]:
def predict_sentiment(text):
    text= preprocess(text)
    vectorized_text= vectorizer.transform([text])
    sentiment= model.predict(vectorized_text)[0]
    return sentiment

text=input("Enter text: ")
sentiment= predict_sentiment(text)

print(f"{text} -> Sentiment: {sentiment}")

this should return a positive output -> Sentiment: positive


# Embedding

In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

from collections import Counter

from nltk.tokenize import word_tokenize

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [68]:
import gc

gc.collect()
torch.cuda.empty_cache()
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using Device:",device)
print(f"Allocated: {torch.cuda.memory_allocated(device) / (1024 ** 2):.2f} MB")
print(f"Cached: {torch.cuda.memory_reserved(device) / (1024 ** 2):.2f} MB")

Using Device: cuda
Allocated: 17.31 MB
Cached: 22.00 MB


## CBOW (Continuous Bag of Words)

In [108]:
corpus = ['The cat sat on the mat',
          'The dog ran in the park',
          'The bird sang in the tree']

corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
print(corpus)

def word2idx(corpus):
    word2idx = {}
    for sentence in corpus:
        for word in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)+1
    return word2idx

word2idx = word2idx(corpus)
print(word2idx)

sequences=[[word2idx[word] for word in sentence] for sentence in corpus]
print(sequences)

[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'ran', 'in', 'the', 'park'], ['the', 'bird', 'sang', 'in', 'the', 'tree']]
{'the': 1, 'cat': 2, 'sat': 3, 'on': 4, 'mat': 5, 'dog': 6, 'ran': 7, 'in': 8, 'park': 9, 'bird': 10, 'sang': 11, 'tree': 12}
[[1, 2, 3, 4, 1, 5], [1, 6, 7, 8, 1, 9], [1, 10, 11, 8, 1, 12]]


In [109]:
vocab_size= len(word2idx) + 1
embedding_size = 10
window_size = 2

contexts=[]
targets=[]

for sequence in sequences:
    for i in range(window_size, len(sequence)-window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)
# print(contexts)
# print(targets)

In [110]:
X = np.array(contexts)
y = np.array(targets)

class CBOWDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts=torch.tensor(contexts, dtype=torch.long)
        self.targets=torch.tensor(targets, dtype=torch.long)
    
    def __len__(self):
        return len(self.targets)
    
    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

dataset= CBOWDataset(X, y)
dataloader= DataLoader(dataset, batch_size=2, shuffle=True)

In [111]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super(CBOWModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.fc = nn.Linear(embedding_size, vocab_size)
    
    def forward(self, inputs):
        embedded = self.embedding(inputs)
        embedded = torch.mean(embedded, dim=1)
        output = self.fc(embedded)
        return output

model = CBOWModel(vocab_size, embedding_size)
model.to(device)

CBOWModel(
  (embedding): Embedding(13, 10)
  (fc): Linear(in_features=10, out_features=13, bias=True)
)

In [112]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)
n_epoch=200
for epoch in range(1,n_epoch+1):
    total_loss= 0
    correct=0
    for context, target in dataloader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad()
        output=model(context)
        loss=loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        correct+=(output.argmax(1)==target).sum().item()
    if epoch%20==0:
        print(f"Epoch: {epoch}\nLoss: {total_loss/len(dataloader)}\nAccuracy: {correct/len(dataloader.dataset)*100}\n----------------------")

Epoch: 20
Loss: 2.366868257522583
Accuracy: 33.33333333333333
----------------------
Epoch: 40
Loss: 2.1170926491419473
Accuracy: 33.33333333333333
----------------------
Epoch: 60
Loss: 1.8913230895996094
Accuracy: 50.0
----------------------
Epoch: 80
Loss: 1.6872553428014119
Accuracy: 66.66666666666666
----------------------
Epoch: 100
Loss: 1.5000263055165608
Accuracy: 83.33333333333334
----------------------
Epoch: 120
Loss: 1.330199917157491
Accuracy: 83.33333333333334
----------------------
Epoch: 140
Loss: 1.1784417231877644
Accuracy: 100.0
----------------------
Epoch: 160
Loss: 1.0436147054036458
Accuracy: 100.0
----------------------
Epoch: 180
Loss: 0.9234066208203634
Accuracy: 100.0
----------------------
Epoch: 200
Loss: 0.8168148001035055
Accuracy: 100.0
----------------------


## skip-gram

In [120]:
corpus = ['The cat sat on the mat',
          'The dog ran in the park',
          'The bird sang in the tree']

corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
print(corpus)

def word2idx(corpus):
    word2idx = {}
    for sentence in corpus:
        for word in sentence:
            if word not in word2idx:
                word2idx[word] = len(word2idx)+1
    return word2idx

word2idx = word2idx(corpus)
print(word2idx)

sequences=[[word2idx[word] for word in sentence] for sentence in corpus]
print(sequences)

[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'ran', 'in', 'the', 'park'], ['the', 'bird', 'sang', 'in', 'the', 'tree']]
{'the': 1, 'cat': 2, 'sat': 3, 'on': 4, 'mat': 5, 'dog': 6, 'ran': 7, 'in': 8, 'park': 9, 'bird': 10, 'sang': 11, 'tree': 12}
[[1, 2, 3, 4, 1, 5], [1, 6, 7, 8, 1, 9], [1, 10, 11, 8, 1, 12]]


In [121]:
vocab_size= len(word2idx) + 1
embedding_size = 10
window_size = 2

contexts=[]
targets=[]

for sequence in sequences:
    for i in range(window_size, len(sequence)-window_size):
        context = sequence[i-window_size:i] + sequence[i+1:i+window_size+1]
        target = sequence[i]
        contexts.append(context)
        targets.append(target)

In [122]:
# using corpus and word2idx from CBOW

def generate_skipgram_data(sequences, window_size):
    contexts = []
    targets = []
    for sequence in sequences:
        for i in range(window_size, len(sequence) - window_size):
            target = sequence[i]
            context_words = sequence[i - window_size:i] + sequence[i + 1:i + window_size + 1]
            for context_word in context_words:
                contexts.append(target)
                targets.append(context_word)
    return np.array(contexts), np.array(targets)

X, y = generate_skipgram_data(sequences, window_size)

In [123]:
class SkipGramDataset(Dataset):
    def __init__(self, contexts, targets):
        self.contexts = torch.tensor(contexts, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.contexts[idx], self.targets[idx]

dataset = SkipGramDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [124]:
class SkipGramModel(nn.Module):
    def __init__(self, vector_size, embedding_size):
        super(SkipGramModel, self).__init__()
        self.embedding = nn.Embedding(vector_size, embedding_size)
        self.fc = nn.Linear(embedding_size, vector_size)

    def forward(self, inputs):
        embedded = self.embedding(inputs) 
        output = self.fc(embedded)
        return output
model=SkipGramModel(vocab_size, embedding_size)
model.to(device)

SkipGramModel(
  (embedding): Embedding(13, 10)
  (fc): Linear(in_features=10, out_features=13, bias=True)
)

In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.01)
n_epoch=200
for epoch in range(1,n_epoch+1):
    total_loss= 0
    # correct=0
    for context, target in dataloader:
        context, target = context.to(device), target.to(device)
        optimizer.zero_grad()
        output=model(context)
        loss=loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
        # correct+=(output.argmax(1)==target).sum().item()
    if epoch%20==0:
        print(f"Epoch: {epoch}\nLoss: {total_loss/len(dataloader)}\n----------------------")

Epoch: 20
Loss: 1.5291657050450642
----------------------
Epoch: 40
Loss: 1.4653369287649791
----------------------
Epoch: 60
Loss: 1.4596364100774128
----------------------
Epoch: 80
Loss: 1.4435964425404866
----------------------
Epoch: 100
Loss: 1.441400796175003
----------------------
Epoch: 120
Loss: 1.4452614684899647
----------------------
Epoch: 140
Loss: 1.4427416026592255
----------------------
Epoch: 160
Loss: 1.4607226848602295
----------------------
Epoch: 180
Loss: 1.4332001010576885
----------------------
Epoch: 200
Loss: 1.4400496035814285
----------------------
